<a href="https://colab.research.google.com/github/vchandraiitk/agentic-ai/blob/main/4_Embedding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip --quiet install langchain-community pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.4/303.4 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 1.6 MB/s eta 0:00:00


In [ ]:
!git clone https://github.com/vchandraiitk/agentic-ai.git

Cloning into 'agentic-ai'...
remote: Enumerating objects: 20, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 20 (delta 6), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (20/20), 4.54 MiB | 10.35 MiB/s, done.
Resolving deltas: 100% (6/6), done.


## Documentation for DocumentLoader
https://python.langchain.com/v0.2/docs/integrations/document_loaders/#all-document-loaders

In [ ]:
import os
os.chdir('/content/agentic-ai')

## Load Text Document

In [ ]:
from langchain_community.document_loaders import TextLoader
text_loader = TextLoader('speech.txt')
text_loader

In [ ]:
text_document = text_loader.load()
# Clean each document's content
for doc in text_document:
    # Replace single newlines with spaces, preserve paragraphs
    import re
    text = doc.page_content
    text = re.sub(r'\n{2,}', '[PARA_BREAK]', text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.replace('[PARA_BREAK]', '\n\n')
    doc.page_content = text
#doc

## Load PDF

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

pdf_loader = PyPDFLoader("Attention.pdf") # Replace 'example.pdf' with your PDF file name
pdf_document = pdf_loader.load()

In [ ]:
#type(pdf_document[0])

## Load text from web URLs

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
import bs4
web_loader = WebBaseLoader(web_paths=(["https://lilianweng.github.io/posts/2023-06-23-agent/", "https://www.ssonetwork.com/intelligent-automation/articles/what-is-agentic-ai"]),
                           bs_kwargs=dict(parse_only=bs4.SoupStrainer(
                               class_=("post-title", "post-content","post-header")

                           )))

In [ ]:
web_doc = web_loader.load()
#web_doc

**RecursiveCharacterTextSplitter** is a utility in LangChain used to split long documents into smaller text chunks. It recursively breaks text by trying different separators, ensuring that:


*   Chunks are within a target size (e.g. 1000 characters).

*   It avoids breaking words or sentences awkwardly when possible.

*   It helps prepare documents for embedding, vector databases, or LLM
input.

**chunk_overlap** ensures context continuity between adjacent chunks, which is crucial for LLMs to understand text coherently across boundaries.

Chunk 1 (chunk_size=100): ...ends with "The CEO announced a new"
Chunk 2: starts with "initiative to expand globally..."

```
Chunk 1 (chunk_size=100): ...ends with "The CEO announced a new"
Chunk 2: starts with "initiative to expand globally..."

```
Now the model has no context that "initiative" is linked to what the CEO said.

```
Chunk 1: ...ends with "The CEO announced a new"
Chunk 2: starts again with "announced a new initiative to expand..."

```
This allows:

**Smoother** transitions

**Better semantic** understanding

More **relevant embeddings** for RAG use cases






In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
final_doc = text_splitter.split_documents(pdf_document)
#final_doc

In [ ]:
#final_doc[0]

In [ ]:
#final_doc[1]

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
final_doc = text_splitter.split_documents(text_document)
#final_doc

In [ ]:
from langchain_text_splitters import HTMLHeaderTextSplitter

html_txt = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Understanding Agentic AI</title>
</head>
<body>
    <header class="post-header">
        <h1 class="post-title">What is Agentic AI?</h1>
        <p class="author">By Jane Doe, April 2024</p>
    </header>

    <section class="post-content">
        <p>Agentic AI refers to artificial intelligence systems that are capable of taking autonomous actions based on goals, planning, and memory.</p>
        <p>Unlike traditional AI, agentic systems can initiate tasks, decide what steps to take, and refine their behavior over time.</p>
    </section>

    <section class="post-content">
        <h2>Why It Matters</h2>
        <p>This approach enables more dynamic and useful interactions, especially in domains like personal assistants, customer service bots, and workflow automation.</p>
    </section>

    <footer class="post-footer">
        <p>Tags: AI, Agentic Systems, LangChain, LLM</p>
    </footer>
</body>
</html>
"""
headers_to_split_on = [
    ("h1", "Header 1"),
    ("h2", "Header 2"),
]

html_splitter = HTMLHeaderTextSplitter(headers_to_split_on)
text = html_splitter.split_text(html_txt)
text

[Document(metadata={'Header 1': 'What is Agentic AI?'}, page_content='What is Agentic AI?'),
 Document(metadata={'Header 1': 'What is Agentic AI?'}, page_content='By Jane Doe, April 2024  \nAgentic AI refers to artificial intelligence systems that are capable of taking autonomous actions based on goals, planning, and memory.  \nUnlike traditional AI, agentic systems can initiate tasks, decide what steps to take, and refine their behavior over time.'),
 Document(metadata={'Header 1': 'What is Agentic AI?', 'Header 2': 'Why It Matters'}, page_content='Why It Matters'),
 Document(metadata={'Header 1': 'What is Agentic AI?', 'Header 2': 'Why It Matters'}, page_content='This approach enables more dynamic and useful interactions, especially in domains like personal assistants, customer service bots, and workflow automation.  \nTags: AI, Agentic Systems, LangChain, LLM')]

## Json Splitter

In [ ]:
import json
import requests
json_data = requests.get("https://api.smith.langchain.com/openapi.json").json()
#json_data

In [ ]:
from langchain_text_splitters import RecursiveJsonSplitter
json_splitter = RecursiveJsonSplitter(max_chunk_size=300)
json_chunks = json_splitter.split_text(json_data)
#json_chunks

In [ ]:
# for json_chunk in json_chunks[:4]:
#     print(json.dumps(json_chunk, indent=4))

In [ ]:
json_doc = json_splitter.create_documents(texts=[json_data])
#json_doc

In [ ]:
# for doc in json_doc[:4]:
#     print(doc.page_content)

In [ ]:
# prompt: mount google drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
os.chdir('/content/drive/MyDrive/Agentic')

In [ ]:
# %%writefile .env
# OPENAI_API_KEY="sk-proj--eXWN61-"
# LANGCHAIN_API_KEY=""
# LANGCHAIN_PROJECT="GenAIWithOpenAI"

Writing .env


In [ ]:
from dotenv import load_dotenv

load_dotenv(".env")
api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
pip --quiet install langchain_openai

In [ ]:
from langchain_openai import OpenAIEmbeddings

In [ ]:
embedding = OpenAIEmbeddings(model="text-embedding-3-large")
query_result = embedding.embed_query("This is just learning class on OpenAI embedding")
query_result

[-0.003396271262317896,
 0.05030474811792374,
 -0.012968536466360092,
 -0.014299720525741577,
 0.02646954543888569,
 0.005104040261358023,
 0.019056951627135277,
 0.05022067576646805,
 0.005163593217730522,
 -0.011406146921217442,
 -0.007398581597954035,
 0.010775585658848286,
 -0.0034943583887070417,
 -0.03253694251179695,
 -0.004519019741564989,
 0.013844314962625504,
 0.013388910330832005,
 0.04024380072951317,
 -0.029342101886868477,
 -0.011483214795589447,
 0.024774037301540375,
 -0.03696488216519356,
 -0.026119234040379524,
 -0.02246198058128357,
 -0.0006870486540719867,
 -0.03225669264793396,
 0.014173608273267746,
 0.052322544157505035,
 -0.01780984364449978,
 0.041336771100759506,
 -0.008561615832149982,
 -0.014895250089466572,
 0.012513130903244019,
 -0.03430251404643059,
 0.00015884442836977541,
 0.020430173724889755,
 0.03214459493756294,
 0.04422333836555481,
 -0.058684203773736954,
 0.013914377428591251,
 0.03248089551925659,
 0.003137040650472045,
 0.008435503579676151,
